# **1. Project Setup**

In [6]:
# Import the libraries used by the local training pipeline.
import os
import pickle
import numpy as np
from tqdm.notebook import tqdm

from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
IMAGE_DIR = PROJECT_ROOT / 'data' / 'Images'
CAPTION_FILE = PROJECT_ROOT / 'data' / 'captions.txt'
WORKING_DIR = PROJECT_ROOT / 'model'

# **2. Dataset Preparation**

In [8]:
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.models import Model

# Change "model" to "vgg_model" here
vgg_model = VGG16(weights="imagenet") 

# Now this line will work perfectly!
feature_extractor = Model(inputs=vgg_model.inputs, outputs=vgg_model.layers[-2].output)

vgg_model.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,357,544 (527.79 MB)

 Trainable params: 138,357,544 (527.79 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
features = {}
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

print("Extracting Image Features...")
for img_name in tqdm(image_files):
    img_path = os.path.join(IMAGE_DIR, img_name)
    image = load_img(img_path, target_size=(224, 224))
    image = img_to_array(image)
    image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
    image = preprocess_input(image) # Zero-centering for VGG
    
    # Extract and store
    feature = feature_extractor.predict(image, verbose=0)
    image_id = img_name.split('.')[0]
    features[image_id] = feature

Extracting Image Features...


  0%|          | 0/8091 [00:00<?, ?it/s]

In [5]:
with open(CAPTION_FILE, 'r') as f:
    next(f) # Skip header
    captions_doc = f.read()

In [10]:
mapping = {}
for line in tqdm(captions_doc.split('\n'), desc="Mapping Captions"):
    if len(line) < 2:
        continue
    tokens = line.split(',')
    image_id, caption = tokens[0].split('.')[0], ",".join(tokens[1:])
    
    if image_id not in mapping:
        mapping[image_id] = []
    mapping[image_id].append(caption.strip())

Mapping Captions:   0%|          | 0/40456 [00:00<?, ?it/s]

# **3. Caption Preprocessing**

# **4. Image Feature Extraction**

In [14]:
import re
def clean_captions(mapping):
    for key, captions in mapping.items():
        for i in range(len(captions)):
            caption = captions[i].lower()
            caption = re.sub(r'[^a-z\s]', '', caption)
            caption = re.sub(r'\s+', ' ', caption).strip()
            words = [word for word in caption.split() if len(word) > 1]
            # Match exactly what you will predict later
            captions[i] = 'startseq ' + " ".join(words) + ' endseq'

In [ ]:
#before preprocess of text
mapping['1000268201_693b08cb0e']

In [16]:
all_captions = [cap for caps in mapping.values() for cap in caps]
max_length = max(len(caption.split()) for caption in all_captions)

In [17]:
# 4. TOKENIZATION & GENERATOR
# ==========================================
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
# CRITICAL FIX: Add 1 for the zero-padding index
vocab_size = len(tokenizer.word_index) + 1

# **5. Vocabulary and Tokenization**

In [18]:
# Train/Test Split
image_ids = list(mapping.keys())
split = int(len(image_ids) * 0.90)
train_keys = image_ids[:split]
test_keys = image_ids[split:]

# **6. Sequence Preparation**

In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np

def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = [], [], []
    n = 0
    while True:
        for key in data_keys:
            n += 1
            captions = mapping[key]
            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    
                    # Ensure flattened feature vector (4096,)
                    X1.append(features[key][0]) 
                    X2.append(in_seq)
                    y.append(out_seq)
            
            if n == batch_size:
                yield (np.array(X1), np.array(X2)), np.array(y)
                X1, X2, y = [], [], []
                n = 0

# **7. Model Architecture**

In [20]:
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, concatenate
from tensorflow.keras.models import Model

# Image pathway
inputs1 = Input(shape=(4096,))
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# Sequence pathway (Removed mask_zero=True)
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256)(inputs2) 
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)

# Decoder pathway (Direct concatenation, no Lambda needed)
decoder1 = concatenate([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

epochs = 50
batch_size = 32
steps_per_epoch = len(train_keys) // batch_size

checkpoint = ModelCheckpoint(
    str(WORKING_DIR / 'model.keras'),
    monitor='loss',
    save_best_only=True,
    mode='min',
    verbose=1
)
reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=2, min_lr=0.0001, verbose=1)

generator = data_generator(train_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size)

model.fit(
    generator,
    epochs=epochs,
    steps_per_epoch=steps_per_epoch,
    callbacks=[checkpoint, reduce_lr],
    verbose=1
)

Starting Training...


c:\Users\trigu\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a generator. The generator is expected to yield already-shuffled data.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 562ms/step - loss: 5.0972
Epoch 1: loss improved from None to 5.09722, saving model to c:\Trigun_Tallapudi\image-caption-generator-main\best_model.keras

Epoch 1: finished saving model to c:\Trigun_Tallapudi\image-caption-generator-main\best_model.keras
227/227 ━━━━━━━━━━━━━━━━━━━━ 129s 563ms/step - loss: 5.0972 - learning_rate: 0.0010
Epoch 2/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 600ms/step - loss: 3.9157
Epoch 2: loss improved from 5.09722 to 3.91568, saving model to c:\Trigun_Tallapudi\image-caption-generator-main\best_model.keras

Epoch 2: finished saving model to c:\Trigun_Tallapudi\image-caption-generator-main\best_model.keras
227/227 ━━━━━━━━━━━━━━━━━━━━ 136s 601ms/step - loss: 3.9157 - learning_rate: 0.0010
Epoch 3/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 598ms/step - loss: 3.4925
Epoch 3: loss improved from 3.91568 to 3.49247, saving model to c:\Trigun_Tallapudi\image-caption-generator-main\best_model.keras

Epoch 3: finished saving model to c

# **8. Model Training**

In [24]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_caption(model, image_feature, tokenizer, max_length):
    in_text = "startseq" # Must match exactly what was added in clean_captions
    
    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        yhat = model.predict([image_feature, sequence], verbose=0)
        yhat = np.argmax(yhat[0])
        word = idx_to_word(yhat, tokenizer)
        
        if word is None:
            break
            
        in_text += " " + word
        
        if word == "endseq": # Must match exactly what was added in clean_captions
            break
            
    return in_text

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

# Validate with test data
actual, predicted = list(), list()

print("Calculating BLEU Scores on Test Set...")
# Changed 'test' to 'test_keys'
for key in tqdm(test_keys):
    captions = mapping[key]

    # Predict the caption using the model
    y_pred = predict_caption(model, features[key], tokenizer, max_length)

    # CLEANING STEP: Remove start/end tags so they don't inflate the score
    actual_captions = [caption.replace("startseq", "").replace("endseq", "").strip().split() for caption in captions]
    y_pred = y_pred.replace("startseq", "").replace("endseq", "").strip().split()

    actual.append(actual_captions)
    predicted.append(y_pred)

# Calculate and print BLEU scores
print("BLEU-1: %f" % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
print("BLEU-2: %f" % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))

# **9. Model Evaluation**

In [ ]:
with (WORKING_DIR / 'tokenizer.pkl').open('wb') as handle:
    pickle.dump(tokenizer, handle)
print('Saved tokenizer.pkl')

Saved tokenizer.pkl and tokenizer_word_index.json


# **10. Model and Tokenizer Export**